In [32]:
# Cell 0 — Project Imports

import nibabel as nib
import numpy as np
import torch

In [33]:
# Cell 1 — Synthetic NIfTI Array와 Affine 생성

# affine = voxel 좌표를 실제 mm 좌표로 Transform

def create_synthetic_nifti(
    array_shape_ijk: tuple[int, int, int],             # 3D 배열 크기
    voxel_spacing_ijk_mm: tuple[float, float, float],  # voxel 한 칸의 실제 mm 크기
    origin_xyz_mm: tuple[float, float, float],         # (0,0,0) voxel의 실제 위치
) -> nib.Nifti1Image:
    """지정한 array shape, spacing과 origin의 synthetic NIfTI 생성"""
    
    # 전체 voxel 개수 = D * H * W
    number_of_voxels = int(
        np.prod(array_shape_ijk)
    )

    # 0 ~ (D x H x W) 값을 만든 뒤
    # (D, H, W) 모양의 3D 배열로 변환
    voxel_array: np.ndarray = np.arange(
        number_of_voxels,
        dtype=np.float32,
    ).reshape(
        array_shape_ijk
    )

    affine: np.ndarray = np.eye(
        4,
        dtype=np.float64,
    ) # [4, 4]
    
    # voxel 한 칸이 실제로 몇 mm인지 affine에 기록
    affine[0, 0] = voxel_spacing_ijk_mm[0]
    affine[1, 1] = voxel_spacing_ijk_mm[1]
    affine[2, 2] = voxel_spacing_ijk_mm[2]
    
    # voxel (0,0,0)이 실제 공간에서 어디 있는지 기록
    affine[:3, 3] = np.asarray(
        origin_xyz_mm,
        dtype=np.float64,
    )

    # 최종 affine:
    # [2.0  0    0   -120]
    # [0   1.5   0    -90]
    # [0    0   3.0   -60]
    # [0    0    0      1]
    
    # 3D voxel 데이터 + 위치정보(affine)를 합쳐서
    # NIfTI 의료영상 객체 생성
    nifti_image = nib.Nifti1Image(
        voxel_array,
        affine,
    )
    
    return nifti_image


# Synthetic volume geometry 정의
synthetic_nifti = create_synthetic_nifti(
    # 3D 배열의 크기
    array_shape_ijk=(
        6,
        8,
        10,
    ),
    # voxel 한 칸의 실제 mm 크기
    voxel_spacing_ijk_mm=(
        2.0,
        1.5,
        3.0,
    ),
    # (0,0,0) voxel의 실제 위치
    origin_xyz_mm=(
        -120.0,
        -90.0,
        -60.0,
    ),
)

# NIfTI container에서 voxel array 추출
# [I=6, J=8, K=10]
nifti_voxel_array: np.ndarray = (
    synthetic_nifti.get_fdata(
        dtype=np.float32,
    )
)  

# Voxel index를 physical coordinate로 변환하는 affine[4, 4] 추출
nifti_affine: np.ndarray = (
    synthetic_nifti.affine
) 

# Header에 기록된 axis별 voxel spacing 추출
header_spacing_ijk_mm = (
    synthetic_nifti.header.get_zooms()[:3]
)

# Affine이 나타내는 physical axis 방향 확인
axis_codes = nib.aff2axcodes(
    nifti_affine
)

print(
    "Array shape:",
    nifti_voxel_array.shape,
)
print(
    "Array dtype:",
    nifti_voxel_array.dtype,
)
print(
    "Header spacing (mm):",
    header_spacing_ijk_mm,
)
print(
    "Axis codes:",
    axis_codes,
)
print("Affine:")
print(nifti_affine)

Array shape: (6, 8, 10)
Array dtype: float32
Header spacing (mm): (np.float32(2.0), np.float32(1.5), np.float32(3.0))
Axis codes: ('R', 'A', 'S')
Affine:
[[   2.     0.     0.  -120. ]
 [   0.     1.5    0.   -90. ]
 [   0.     0.     3.   -60. ]
 [   0.     0.     0.     1. ]]


In [34]:
# Cell 2 — Voxel Index와 Physical Coordinate 구분

# physical = origin + (index * spacing)

synthetic_nifti = create_synthetic_nifti(
    # 3D 배열 크기    
    array_shape_ijk=(
        6,
        8,
        10,
    ),
    # voxel 한 칸의 실제 mm 크기
    voxel_spacing_ijk_mm=(
        2.0,
        1.5,
        3.0,
    ),
    # (0,0,0) voxel의 실제 위치
    origin_xyz_mm=(
        -120.0,
        -90.0,
        -60.0,
    ),
)

# NIfTI 객체에서 실제 voxel array 추출
nifti_voxel_array: np.ndarray = (
    
    # 예시 데이터: [I=6, J=8, K=10]
    synthetic_nifti.get_fdata(
        dtype=np.float32,
    )
)

# 확인할 voxel의 array index 선택
selected_voxel_index_ijk: tuple[
    int,
    int,
    int,
] = (
    2,
    3,
    4,
)

selected_voxel_value = nifti_voxel_array[
    selected_voxel_index_ijk
].item()


# Voxel index를 계산용 Tensor로 변환
voxel_index_ijk = torch.tensor(
    selected_voxel_index_ijk,
    dtype=torch.float64,
)  # [3]

# 각 array axis에서 voxel 한 칸의 실제 이동 거리
voxel_spacing_ijk_mm = torch.tensor(
    [
        2.0,
        1.5,
        3.0,
    ],
    dtype=torch.float64,
)  # [3]

# Voxel index (0,0,0)의 physical coordinate
physical_origin_xyz_mm = torch.tensor(
    [
        -120.0,
        -90.0,
        -60.0,
    ],
    dtype=torch.float64,
)  # [3]

# 실제 물리적 좌표를 axis별로 직접 계산
physical_coordinate_xyz_mm = (
    physical_origin_xyz_mm
    + voxel_index_ijk
    * voxel_spacing_ijk_mm
)


# I index만 1 증가한 이웃 voxel 선택
neighbor_voxel_index_ijk = torch.tensor(
    [
        3.0,
        3.0,
        4.0,
    ],
    dtype=torch.float64,
) # [3]

# 이웃 voxel의 physical coordinate 계산
neighbor_physical_coordinate_xyz_mm = (
    physical_origin_xyz_mm
    + neighbor_voxel_index_ijk
    * voxel_spacing_ijk_mm
)  # [3]


# 두 voxel 사이의 index 차이 계산
# 결과: [1, 0, 0]
index_difference_ijk = (
    neighbor_voxel_index_ijk
    - voxel_index_ijk
)  # [3]


# 두 voxel 사이의 실제 mm 거리 차이 계산
# 결과: [2.0, 0.0, 0.0] mm
physical_difference_xyz_mm = (
    neighbor_physical_coordinate_xyz_mm
    - physical_coordinate_xyz_mm
)  # [3]



print(
    "Voxel array shape:",
    nifti_voxel_array.shape,
)
print(
    "Selected voxel index (IJK):",
    selected_voxel_index_ijk,
)
print(
    "Selected voxel value:",
    selected_voxel_value,
)
print()

print(
    "Physical origin (XYZ mm):",
    physical_origin_xyz_mm,
)
print(
    "Voxel spacing (IJK mm): ",
    voxel_spacing_ijk_mm,
)
print(
    "Physical coordinate:",
    physical_coordinate_xyz_mm,
)
print()

print(
    "Index difference:   ",
    index_difference_ijk,
)
print(
    "Physical difference:",
    physical_difference_xyz_mm,
)

Voxel array shape: (6, 8, 10)
Selected voxel index (IJK): (2, 3, 4)
Selected voxel value: 194.0

Physical origin (XYZ mm): tensor([-120.,  -90.,  -60.], dtype=torch.float64)
Voxel spacing (IJK mm):  tensor([2.0000, 1.5000, 3.0000], dtype=torch.float64)
Physical coordinate: tensor([-116.0000,  -85.5000,  -48.0000], dtype=torch.float64)

Index difference:    tensor([1., 0., 0.], dtype=torch.float64)
Physical difference: tensor([2., 0., 0.], dtype=torch.float64)


In [35]:
# Cell 3 — Affine 기반 Index-to-Physical 변환

# Voxel indices                [N,3]
# Homogeneous ones             [N,1]
#          ↓ torch.cat(dim=1)
# Homogeneous voxel indices    [N,4]
#          ↓ @ affine.T [4,4]
# Homogeneous physical coords  [N,4]
#          ↓ [:,:3]
# Physical coordinates         [N,3]


def voxel_indices_to_physical_coordinates(
    voxel_indices_ijk: torch.Tensor,  # [N, 3]
    affine_ijk_to_xyz: torch.Tensor,  # [4, 4]
) -> torch.Tensor:                    # [N, 3]
    """Voxel indices를 affine 기반 physical coordinates로 변환"""
    
    number_of_voxels = (
        voxel_indices_ijk.shape[0] # N
    )
    
    # [N, 1]
    homogeneous_ones = torch.ones(
        (
            number_of_voxels,
            1,
        ),
        dtype=voxel_indices_ijk.dtype,
        device=voxel_indices_ijk.device,
    )
    
    # [N, 3] + [N, 1] -> [N, 4]
    homogeneous_voxel_indices = torch.cat(
        (
            voxel_indices_ijk, # [N, 3]
            homogeneous_ones,  # [N, 1]
        ),
        dim=1,
    )
    
    # physical_h = affine @ voxel_h
    # 
    # voxel_indices: [N, 4]
    # affine matrix: [4, 4]
    # [N, 4] @ [4, 4].T -> [N, 4]
    homogeneous_physical_coordinates = (
        homogeneous_voxel_indices
        @ affine_ijk_to_xyz.T
    )
    
    # 마지막 homogeneous coordinate 제거:
    # [N, 4] -> [N, 3]
    physical_coordinates_xyz_mm = (
        homogeneous_physical_coordinates[
            :,
            :3,
        ]
    )
    
    return physical_coordinates_xyz_mm




voxel_indices_ijk = torch.tensor(
    [
        [0.0, 0.0, 0.0], # Volume origin voxel
        [2.0, 3.0, 4.0], # Cell 2에서 검사한 voxel
        [3.0, 3.0, 4.0], # I축으로 한 칸 이동한 이웃 voxel
    ],
    dtype=torch.float64,
)  # [N=3, 3]


# NIfTI의 NumPy affine을 PyTorch Tensor로 변환
# 
# affine matrix[4, 4]:
# [2.0  0    0   -120]
# [0   1.5   0    -90]
# [0    0   3.0   -60]
# [0    0    0      1]
affine_ijk_to_xyz = torch.from_numpy(
    nifti_affine
).to(
    dtype=torch.float64,
)


# voxel index를 physical coordinates로 일괄 변환
physical_coordinates_xyz_mm = (
    voxel_indices_to_physical_coordinates(
        voxel_indices_ijk=voxel_indices_ijk,
        affine_ijk_to_xyz=affine_ijk_to_xyz,
    )
)  # [N=3, 3]


# Nibabel 공식 구현으로 동일한 좌표 변환 후, Tensor로 변환
nibabel_coordinates_xyz_mm = (
    nib.affines.apply_affine(
        nifti_affine,
        voxel_indices_ijk.cpu().numpy(),
    )
)  # [N=3, 3]
nibabel_coordinates_tensor = torch.from_numpy(
    nibabel_coordinates_xyz_mm
).to(
    dtype=torch.float64,
)  # [N=3, 3]


# Scratch 구현과 nibabel 공식 구현의 전체 좌표 비교
coordinates_match = torch.allclose(
    physical_coordinates_xyz_mm,
    nibabel_coordinates_tensor,
)


print(
    "Voxel indices shape:",
    voxel_indices_ijk.shape,
)
print(
    "Affine shape:",
    affine_ijk_to_xyz.shape,
)
print(
    "Physical coordinates shape:",
    physical_coordinates_xyz_mm.shape,
)
print()

print("Voxel indices (IJK):")
print(voxel_indices_ijk)
print()

print("Scratch physical coordinates (XYZ mm):")
print(physical_coordinates_xyz_mm)
print()

print("Nibabel physical coordinates (XYZ mm):")
print(nibabel_coordinates_tensor)
print()

print(
    "Scratch matches nibabel:",
    coordinates_match,
)


Voxel indices shape: torch.Size([3, 3])
Affine shape: torch.Size([4, 4])
Physical coordinates shape: torch.Size([3, 3])

Voxel indices (IJK):
tensor([[0., 0., 0.],
        [2., 3., 4.],
        [3., 3., 4.]], dtype=torch.float64)

Scratch physical coordinates (XYZ mm):
tensor([[-120.0000,  -90.0000,  -60.0000],
        [-116.0000,  -85.5000,  -48.0000],
        [-114.0000,  -85.5000,  -48.0000]], dtype=torch.float64)

Nibabel physical coordinates (XYZ mm):
tensor([[-120.0000,  -90.0000,  -60.0000],
        [-116.0000,  -85.5000,  -48.0000],
        [-114.0000,  -85.5000,  -48.0000]], dtype=torch.float64)

Scratch matches nibabel: True


In [36]:
# Cell 4 — Inverse Affine 기반 Physical-to-Index 복원

# Forward:
# original voxel indices
#       ↓ affine
# physical coordinates
#
# Backward:
# physical coordinates
#       ↓ inverse affine
# recovered voxel indices

def physical_coordinates_to_voxel_indices(
    physical_coordinates_xyz_mm: torch.Tensor,   # [N, 3]
    affine_ijk_to_xyz: torch.Tensor,             # [4, 4]
) -> torch.Tensor:                               # [N, 3]
    """Physical coordinates를 inverse affine 기반 voxel indices로 변환"""

    number_of_coordinates = (
        physical_coordinates_xyz_mm.shape[0] # N
    )

    # Inverse of affine matrix: [4, 4]
    inverse_affine_xyz_to_ijk = (
        torch.linalg.inv(
            affine_ijk_to_xyz
        )
    ) 

    # [N, 1]
    homogeneous_ones = torch.ones(
        (
            number_of_coordinates,
            1,
        ),
        dtype=physical_coordinates_xyz_mm.dtype,
        device=physical_coordinates_xyz_mm.device,
    ) 

    # Shape:
    # [N, 3] + [N, 1] → [N, 4]
    homogeneous_physical_coordinates = torch.cat(
        (
            physical_coordinates_xyz_mm,
            homogeneous_ones,
        ),
        dim=1,
    )

    # voxel_h = inverse_affine @ physical_h
    #
    # inverse_affine: [4, 4]
    # physical_h    : [N, 4]
    # [N, 4] @ [4, 4].T -> [N, 4]
    homogeneous_voxel_indices = (
        homogeneous_physical_coordinates
        @ inverse_affine_xyz_to_ijk.T
    )

    # 마지막 homogeneous coordinate 제거
    # [N, 4] → [N, 3]
    voxel_indices_ijk = (
        homogeneous_voxel_indices[
            :,
            :3,
        ]
    )

    return voxel_indices_ijk



recovered_voxel_indices_ijk = (
    physical_coordinates_to_voxel_indices(
        physical_coordinates_xyz_mm=(
            physical_coordinates_xyz_mm
        ),  # [N=3, 3]
        affine_ijk_to_xyz=(
            affine_ijk_to_xyz
        ),  # [4, 4]
    )
)  # [N=3, 3]


# 원본 index와 복원 index의 절대 오차 계산
round_trip_absolute_error = torch.abs(
    recovered_voxel_indices_ijk
    - voxel_indices_ijk
)  # [N=3, 3]

maximum_round_trip_error = (
    round_trip_absolute_error
    .max()
    .item()
)


# Floating-point voxel coordinate를
# 실제 discrete array index로 사용할 경우 반올림
rounded_voxel_indices_ijk = torch.round(
    recovered_voxel_indices_ijk
).to(
    dtype=torch.long,
)  # [N=3, 3]


# Nibabel 공식 inverse affine 결과 계산
inverse_nifti_affine = np.linalg.inv(
    nifti_affine
)  # [4, 4]

nibabel_recovered_indices_ijk = (
    nib.affines.apply_affine(
        inverse_nifti_affine,
        physical_coordinates_xyz_mm
        .cpu()
        .numpy(),
    )
)  # [N=3, 3]
# Nibabel 결과를 PyTorch Tensor로 변환
nibabel_recovered_indices_tensor = (
    torch.from_numpy(
        nibabel_recovered_indices_ijk
    ).to(
        dtype=torch.float64,
    )
)  # [N=3, 3]


# Scratch inverse와 nibabel inverse 결과 비교
inverse_results_match = torch.allclose(
    recovered_voxel_indices_ijk,
    nibabel_recovered_indices_tensor,
)


# Voxel 중심 사이에 있는 physical coordinate 생성
#
# Physical coordinate:
# [-115.0, -85.5, -48.0] mm
#
# X 방향 계산:
# i = (-115 - (-120)) / 2
#   = 5 / 2
#   = 2.5
#
# 따라서 discrete voxel 중심이 아닌
# continuous voxel coordinate [2.5, 3.0, 4.0] 예상
between_voxel_physical_coordinate = torch.tensor(
    [
        [
            -115.0,
            -85.5,
            -48.0,
        ]
    ],
    dtype=torch.float64,
)  # [N=1, 3]


# Voxel 사이의 physical point를 continuous index로 변환
between_voxel_index_ijk = (
    physical_coordinates_to_voxel_indices(
        physical_coordinates_xyz_mm=(
            between_voxel_physical_coordinate
        ),  # [N=1, 3]
        affine_ijk_to_xyz=(
            affine_ijk_to_xyz
        ),  # [4, 4]
    )
)  # [N=1, 3]


print("Forward affine:")
print(affine_ijk_to_xyz)
print()

print("Inverse affine:")
print(
    torch.linalg.inv(
        affine_ijk_to_xyz
    )
)
print()

print("Original voxel indices:")
print(voxel_indices_ijk)
print()

print("Recovered voxel indices:")
print(recovered_voxel_indices_ijk)
print()

print(
    "Rounded voxel indices:",
)
print(rounded_voxel_indices_ijk)
print()

print(
    "Maximum round-trip error:",
    maximum_round_trip_error,
)
print(
    "Scratch inverse matches nibabel:",
    inverse_results_match,
)
print()

print(
    "Between-voxel physical coordinate:",
)
print(between_voxel_physical_coordinate)
print(
    "Continuous voxel index:",
)
print(between_voxel_index_ijk)

Forward affine:
tensor([[   2.0000,    0.0000,    0.0000, -120.0000],
        [   0.0000,    1.5000,    0.0000,  -90.0000],
        [   0.0000,    0.0000,    3.0000,  -60.0000],
        [   0.0000,    0.0000,    0.0000,    1.0000]], dtype=torch.float64)

Inverse affine:
tensor([[ 0.5000,  0.0000,  0.0000, 60.0000],
        [ 0.0000,  0.6667,  0.0000, 60.0000],
        [ 0.0000,  0.0000,  0.3333, 20.0000],
        [ 0.0000,  0.0000,  0.0000,  1.0000]], dtype=torch.float64)

Original voxel indices:
tensor([[0., 0., 0.],
        [2., 3., 4.],
        [3., 3., 4.]], dtype=torch.float64)

Recovered voxel indices:
tensor([[0., 0., 0.],
        [2., 3., 4.],
        [3., 3., 4.]], dtype=torch.float64)

Rounded voxel indices:
tensor([[0, 0, 0],
        [2, 3, 4],
        [3, 3, 4]])

Maximum round-trip error: 0.0
Scratch inverse matches nibabel: True

Between-voxel physical coordinate:
tensor([[-115.0000,  -85.5000,  -48.0000]], dtype=torch.float64)
Continuous voxel index:
tensor([[2.5000, 3.0